In [1]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def create_dataset(n_samples=1000, noise=0.2, seed=42):
    g = torch.Generator().manual_seed(seed)
    x = torch.linspace(-1.0, 1.0, n_samples).unsqueeze(1)
    y = 2.0 * x + 3.0 + noise * torch.randn_like(x, generator=g)
    # train/val split
    n_train = int(n_samples * 0.8)
    x_train, y_train = x[:n_train], y[:n_train]
    x_val, y_val = x[n_train:], y[n_train:]
    return (TensorDataset(x_train, y_train), TensorDataset(x_val, y_val))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def main():
    device = get_device()
    print(f"device: {device}")

    train_ds, val_ds = create_dataset()
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256)

    model = MLP().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

    epochs = 50
    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        if ep % 10 == 0 or ep == 1 or ep == epochs:
            print(f"epoch {ep:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

    # save
    path = "pytorch_sample_model.pth"
    torch.save(model.state_dict(), path)
    print(f"saved: {path}")

    # inference demo
    model.eval()
    with torch.no_grad():
        x_new = torch.tensor([[4.0]], device=device)
        y_pred = model(x_new).item()
    print(f"x=4.0 -> pred≈{y_pred:.4f} (true≈11.0)")

if __name__ == "__main__":
    main()


device: cuda


TypeError: randn_like() got an unexpected keyword argument 'generator'